In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---


# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 2 &mdash; Signals, Systems and Processing</h1></center>

A seismometer, a gravimeter and a ground-penetrating radar all produce the same kind of object: numbers recorded at regular steps of time or
position. Before we can analyze those numbers we need a precise vocabulary for what they are (signals), how they were made (continuous,
discrete and digital), and what we do to them (systems and processing). This module sets out that vocabulary with examples from the data
sets used throughout the course, and ends by separating a recorded signal from the random process that generated it: the first appearance
of noise, which much of this course is about removing.

[Digital Signal Processing](https://en.wikipedia.org/wiki/Digital_signal_processing) (or DSP) is one of the *most powerful technologies* that will shape science and engineering in the twenty-first century.  Revolutionary changes have already been made in a broad range of fields: communications, medical imaging, radar & sonar, high fidelity music reproduction, and oil prospecting, to name just a few.  

Each of these areas has developed a deep DSP technology, with its own algorithms, mathematics, and specialized techniques.  This combination of breadth and depth makes it *impossible* for any one individual to master all of the DSP technology that has been developed.   DSP education involves two tasks: 

* learning general concepts that apply to the field as a whole; and
* learning specialized techniques for your particular area of interest. 

The purpose of this module is to provide you with some of the key terminology that we will be covering in this DSP course: **signals**; **continuous, discrete** and **digital**; **systems**; and **processing**.

# Outline
- [2.1 Signals](#2.1-Signals)
- [2.2 Continuous, Discrete and Digital Signals](#2.2-Continuous,-Discrete-and-Digital-Signals)
- [2.3 Systems](#2.3-Systems)
- [2.4 Processing](#2.4-Processing)
- [2.5 Signals versus Underlying Processes](#2.5-Signals-versus-Underlying-Processes)
- [2.6 DSP versus Analog Signal Processing](#2.6-DSP-versus-Analog-Signal-Processing)
- [Microquestions](#Microquestions)

<a name="2.1"></a>
## 2.1 Signals

A signal is anything that conveys **information** and is a description of how one (or a set of) **parameter(s)** relates to another parameter(s) (e.g., amplitude or voltage as a function of time).  Examples of signals are everywhere, including:
* Seismic or radar pulse
* Speech
* DNA sequence
* Stock price
* Image
* Video

A signal can have single or multiple independent variables. For example, you'll be familiar with the following examples:
* 1D - Speech: $s=s(t)$
* 2D - Image: $I=I(x,y)$, Topography map: $h=h(\phi,\lambda)$ (elevation vs latitude and longitude)
* 3D - 3D Seismic/GPR shot gather: $S = S(t,r_x,r_y)$
* 4D - EM/Seismic wavefield: $W=W(t,x,y,z)$
* 5D - 3D Seismic data set: $D=D(t,r_x,r_y,s_x,s_y)$

Figure 2.1 shows a 1D and a 2D example from the data sets used in this course.

In [ ]:
import read_in_rsf as sf          # RSF reader in the repository root (for data/CAPshot)
C = DSP_COLORS
# (a) 1D signal: Tohoku earthquake, vertical velocity at GR.WET (Germany), 20 samples/s
toh = np.load('data/Tohoku.npz')['tohoku'].astype(float)*1000      # m/s -> mm/s
fs_toh = 20.0
blk = 200                                                           # plot the min/max envelope of 10 s blocks
nb = len(toh)//blk
env = toh[:nb*blk].reshape(nb, blk)
tb = (np.arange(nb) + 0.5)*blk/fs_toh/3600                          # hours after 05:00 UTC
# (b) 2D signal: CAPshot shot gather, 2500 samples x 300 traces, 4 ms
a = sf.read_rsf_file('data/CAPshot', use_memmap=True)
dt = a.view.dx[0]; nt, nx = a.view.nx
shot = np.array(a.data, dtype=float)
nt = int(round(4.0/dt)) + 1                                         # plot only the first 4 s (Jeff, 2026-10-05)
shot = shot[:nt]
shot[np.abs(shot) > 1e3*np.median(np.abs(shot))] = 0.0             # a few huge near-source spikes zeroed
gath = shot/(np.sqrt(np.mean(shot**2, axis=0, keepdims=True)) + 1e-30)
clip = np.percentile(np.abs(gath), 98)
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw=dict(width_ratios=[1.3, 1]))
axa.fill_between(tb, env.min(axis=1), env.max(axis=1), color=C['time_domain'], lw=0)
axa.set_xlim(0, len(toh)/fs_toh/3600); axa.set_xlabel('time after 05:00 UTC (h)'); axa.set_ylabel('vertical velocity (mm/s)')
axa.set_title('(a) 1D signal $s(t)$: Tohoku earthquake at GR.WET', fontsize=12)
axb.imshow(gath, cmap='gray', vmin=-clip, vmax=clip, aspect='auto', extent=[0.5, nx + 0.5, (nt - 1)*dt, 0])
axb.set_xlabel('trace number $r$'); axb.set_ylabel('time $t$ (s)'); axb.grid(False)
axb.set_title('(b) 2D signal $S(t, r)$: CAPshot gather', fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 2.1.</b> Two geophysical signals used throughout the course. (a) A 1D signal: the vertical ground velocity (mm/s) of the 2011 Tohoku earthquake recorded at station GR.WET in Germany, 20 samples per second for 6.0 h (plotted as the minimum/maximum in 10 s blocks). (b) A 2D signal: the first 4 s (1001 of 2500 samples at 4 ms) of the CAPshot shot gather, 300 traces, each trace scaled to unit RMS. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; geophysical data as the running example.</b> Figure 2.1 shows the two data sets used most in this course: a 1D signal
$s(t)$ and a 2D signal $S(t, r)$. The CASC93 array of Module 1 recorded $S(t, r, c)$, with three components $c$ at each of 40 stations, and the
gravity and magnetic maps of GPGN 328 (POTE1) are 2D signals $g(x, y)$ sampled in space rather than time. Counting the independent variables
is the first step in choosing how to process any of them.
</div>

<a name="2.2"></a>
## 2.2 Continuous, Discrete and Digital Signals

There are three types of signals that are functions of *time*:
1. **Continuous-time** (analog) - $x(t)$: defined on a continuous range of time *t*, amplitude can be any value.
2. **Discrete-time** - $x[n] = x(nT)$: defined only at discrete instants of time $t=\dots,-T,0,T,2T,\dots$, where the amplitude can be any value. We write $x[n]$ for the $n$-th sample of $x(t)$ taken every $T$ seconds.
3. **Digital** (quantized) - $x_Q[n]$: both time and amplitude are discrete. Signals only defined at $t=...,-T,0,T,2T,...$ and amplitude is confined to a finite set of numbers.

Figure 2.2 illustrates the three types.

In [ ]:
from matplotlib.patches import FancyBboxPatch
C = DSP_COLORS

def signal_types_figure():
    """Analog -> sampled -> quantized (Modules 2 and 9 share this code). x(t) = 2cos(2pi(t-1.5)/20), T = 1, Delta = 1."""
    x = lambda t: 2*np.cos(2*np.pi*(t - 1.5)/20)
    T, Delta = 1.0, 1.0
    n = np.arange(-1, 12)
    tt = np.linspace(-1.5, 11.5, 600)
    xn = x(n*T)
    xq = Delta*np.rint(xn/Delta) + 0.0
    fig = plt.figure(figsize=(13, 6.8))
    gs = fig.add_gridspec(2, 3, height_ratios=[0.8, 1])
    # --- top row: the chain x(t) -> sampler -> quantizer -> DSP
    ax0 = fig.add_subplot(gs[0, :]); ax0.axis('off'); ax0.set_xlim(0, 13); ax0.set_ylim(0, 2.2)
    boxes = [(2.2, 'Sampler\n$t = nT$'), (5.6, 'Quantizer\n$Q(x)$'), (9.0, 'DSP\n(computer)')]
    for xc, lab in boxes:
        ax0.add_patch(FancyBboxPatch((xc - 1.0, 0.45), 2.0, 1.3, boxstyle='round,pad=0.05', fc='white', ec=C['time_domain'], lw=2))
        ax0.text(xc, 1.1, lab, ha='center', va='center', fontsize=12)
    # small staircase icon inside the quantizer, drawn from Q itself
    s = np.linspace(-2.2, 2.2, 200); qs = Delta*np.rint(s/Delta)
    ax0.plot(5.6 + 0.62 + 0.12*s, 1.1 + 0.12*qs, color=C['sampled'], lw=1.2)
    arrows = [(0.15, 1.2, '$x(t)$'), (3.2, 4.6, '$x[n]$'), (6.6, 8.0, '$x_Q[n]$'), (10.0, 12.6, 'output')]
    for x0_, x1_, lab in arrows:
        ax0.annotate('', xy=(x1_, 1.1), xytext=(x0_, 1.1), arrowprops=dict(arrowstyle='-|>', lw=1.8, color='k'))
        ax0.text((x0_ + x1_)/2, 1.45, lab, ha='center', va='bottom', fontsize=12)
    # --- bottom row
    titles = ['(a) continuous-time $x(t)$', '(b) discrete-time $x[n] = x(nT)$', '(c) digital $x_Q[n]$ ($\\Delta = 1$)']
    for j in range(3):
        ax = fig.add_subplot(gs[1, j])
        ax.axhline(0, color='k', lw=0.8)
        if j == 0:
            ax.plot(tt, x(tt), color=C['continuous'], lw=2)
        else:
            ax.plot(tt, x(tt), color=C['continuous'], lw=1, alpha=0.35)
            vals = xn if j == 1 else xq
            ml, sl, bl = ax.stem(n*T, vals, basefmt=' ')
            plt.setp(sl, color=C['sampled'], lw=1.5); plt.setp(ml, color=C['sampled'], ms=6)
            if j == 2:
                for lev in (-2, -1, 0, 1, 2):
                    ax.axhline(lev*Delta, ls='--', color=C['neutral'], lw=0.8)
        ax.set_xlim(-1.6, 11.6); ax.set_ylim(-2.4, 2.4)
        ax.set_xlabel('$t$ (s)' if j == 0 else '$n$ (samples, $T = 1$ s)'); ax.set_title(titles[j], fontsize=12)
        if j == 0:
            ax.set_ylabel('amplitude')
    plt.tight_layout()
    return fig

signal_types_figure()
plt.show()

<b>Figure 2.2.</b> Continuous-time, discrete-time and digital signals. Top: the analog signal $x(t)$ passes through a sampler ($t = nT$) and a quantizer $Q(x)$ before it reaches the computer. Bottom: (a) $x(t) = 2\cos(2\pi(t-1.5)/20)$; (b) its samples $x[n] = x(nT)$ for $T = 1$ s; (c) the samples rounded to the five levels $-2, \dots, 2$ (step $\Delta = 1$, dashed), with a largest rounding error of 0.414 $\le \Delta/2$. Module 9 uses the same figure. <i>(Python-generated.)</i>

In DSP we deal with $x_Q[n]$ because this corresponds with computer-based processing (which is quantized by definition - e.g. 16-bit vs 32-bit system).  In this course we will assume that **discrete-time signal** is equivalent to a **digital signal** (equivalent to saying that the quantizer has infinite resolution).  Thus, we will commonly write continuous and discrete (and quantized) signals as $x(t)$ and $x[n]$, where parentheses and square brackets will denote continuity and discreteness, respectively.

<div class="alert alert-info">
<b>Note:</b> A discrete-time signal $x[n]$ is a sequence in the MATH 213 sense, indexed by the integers. Nothing in the sequence itself records
the spacing $T$; that has to be carried alongside the data (a sample rate in a file header, or <code>dt</code> in code).
</div>

<a name="2.3"></a>
## 2.3 Systems

A **system** is a mathematical model or abstraction of a physical process that relates **input** to **output**.  Any system that processes digital signals is called a **digital system** or **digital signal processor**.

Examples include:
* Amplifier 
    * input: ${\rm cos}\,\omega t$
    * output: $10\,{\rm cos}\,\omega t$
* Delay 
    * input: $f[n]$
    * output: $g[n] = f[n-p]$, where the integer $p>0$ is the delay in samples
* Feature extraction 
    * Input: ${\rm cos}\,\omega_1 t + {\rm cos}\,\omega_2 t$
    * Output: [$\omega_1,\omega_2$]
* Seismometer 
    * Input: ground velocity
    * Output: voltage
* Digitizer 
    * Input: voltage
    * Output: integers (Figure 2.2)

<div class="alert alert-info">
<b>Synthesis &mdash; systems as operators.</b> A system maps a whole input function to a whole output function. The RC low-pass circuit of PHGN 200 is the
standard analog example: the input voltage $V_{in}(t)$ and the capacitor voltage $V_{out}(t)$ are linked by the first-order ODE
$RC\,dV_{out}/dt + V_{out} = V_{in}$ from MATH 225. Figure 2.3 shows this circuit and a few lines of code doing the same job.
</div>

<a name="2.4"></a>
## 2.4 Processing

**Processing** performs a particular function by passing a signal through a **system**. Examples include:

* analog processing of an **analog** signal, and
* digital processing of an analog signal.

Figure 2.3 compares the two for the same task: removing a 12 Hz component from a 1.5 Hz signal.

In [ ]:
from matplotlib.patches import FancyBboxPatch
C = DSP_COLORS
xs = lambda t: np.sin(2*np.pi*1.5*t) + 0.3*np.sin(2*np.pi*12*t)     # 1.5 Hz signal + 12 Hz "noise"
fc = 3.0; tau = 1/(2*np.pi*fc)                                      # RC low-pass, cutoff 3 Hz
# analog RC output: RC dV/dt + V = V_in, stepped exactly on a fine (10 us) grid
h = 1e-5; t = np.arange(0, 2 + h/2, h); xa = xs(t); ya = np.zeros_like(t); a = np.exp(-h/tau)
for k in range(1, len(t)):
    ya[k] = a*ya[k-1] + (1 - a)*xa[k]
# digital: sample at 100 Hz, one-pole filter y[n] = y[n-1] + alpha (x[n] - y[n-1]), linear-interpolation DAC
fs = 100.0; dtn = 1/fs; alpha = dtn/(tau + dtn)
tn = np.arange(0, 2 + dtn/2, dtn); xn = xs(tn); yn = np.zeros_like(tn)
for k in range(1, len(tn)):
    yn[k] = yn[k-1] + alpha*(xn[k] - yn[k-1])
yd = np.interp(t, tn, yn)

fig, ax = plt.subplots(figsize=(13.5, 5.2))
fig.subplots_adjust(left=0.005, right=0.995, bottom=0.01, top=0.99)
ax.set_xlim(0, 13.4); ax.set_ylim(0.85, 6.15); ax.axis('off')
def box(x0, x1, yc, text):
    ax.add_patch(FancyBboxPatch((x0, yc - 0.55), x1 - x0, 1.1, boxstyle='round,pad=0.04', fc='white', ec=C['time_domain'], lw=2))
    ax.text((x0 + x1)/2, yc, text, ha='center', va='center', fontsize=11)
def arrow(x0, x1, yc):
    ax.annotate('', xy=(x1, yc), xytext=(x0, yc), arrowprops=dict(arrowstyle='-|>', lw=1.6, color='k'))
def inset(x0, yc, title, draw):
    ia = ax.inset_axes([x0, yc - 0.65, 1.6, 1.3], transform=ax.transData)
    draw(ia); ia.set_xticks([]); ia.set_yticks([]); ia.set_xlim(0, 0.5); ia.set_ylim(-1.5, 1.5)
    ia.set_title(title, fontsize=10, pad=3)
    return ia
win = lambda tt: (tt >= 0.6) & (tt <= 1.1)                          # show 0.6-1.1 s in each inset
def cont(ia, tt, yy, col):
    m = win(tt); ia.plot(tt[m] - 0.6, yy[m], color=col, lw=1.4)
def stems(ia, tt, yy, col):
    m = win(tt); ia.vlines(tt[m] - 0.6, 0, yy[m], color=col, lw=0.8); ia.plot(tt[m] - 0.6, yy[m], '.', color=col, ms=3)
# (a) analog chain
yA = 4.6
ax.text(0.05, 5.9, '(a) Analog processing of an analog signal', fontsize=12, fontweight='bold')
inset(0.1, yA, 'analog input $x(t)$', lambda ia: cont(ia, t, xa, C['continuous']))
arrow(1.8, 5.3, yA); box(5.3, 8.1, yA, 'analog RC low-pass\n$RC\\,\\dot V_{out} + V_{out} = V_{in}$'); arrow(8.1, 11.6, yA)
inset(11.7, yA, 'analog output $y(t)$', lambda ia: cont(ia, t, ya, C['output']))
# (b) digital chain
yB = 1.6
ax.text(0.05, 2.85, '(b) Digital processing of an analog signal', fontsize=12, fontweight='bold')
inset(0.1, yB, 'analog input $x(t)$', lambda ia: cont(ia, t, xa, C['continuous']))
arrow(1.8, 2.15, yB); box(2.15, 3.45, yB, 'ADC\n$f_s = 100$ Hz'); arrow(3.45, 3.8, yB)
inset(3.85, yB, '$x[n]$', lambda ia: stems(ia, tn, xn, C['sampled']))
arrow(5.5, 5.85, yB); box(5.85, 7.55, yB, 'digital filter\n(one pole)'); arrow(7.55, 7.9, yB)
inset(7.95, yB, '$y[n]$', lambda ia: stems(ia, tn, yn, C['output']))
arrow(9.6, 9.95, yB); box(9.95, 11.25, yB, 'DAC'); arrow(11.25, 11.6, yB)
def final(ia):
    cont(ia, t, ya, C['neutral']); cont(ia, t, yd, C['output'])
inset(11.7, yB, 'output (gray: analog)', final)
plt.show()

<b>Figure 2.3.</b> Two ways to low-pass filter the same analog signal, $x(t) = \sin(2\pi\,1.5t) + 0.3\sin(2\pi\,12t)$ (each inset shows 0.6–1.1 s). (a) Analog processing: an RC circuit with cutoff $f_c = 3$ Hz ($\tau = RC = 0.0531$ s) passes the 1.5 Hz component with gain 0.894 and reduces the 12 Hz component to 0.243. (b) Digital processing: an analog-to-digital converter (ADC) samples $x(t)$ at 100 Hz, the computer applies the one-line filter $y[n] = y[n-1] + \alpha(x[n] - y[n-1])$ with $\alpha = 0.159$, and a digital-to-analog converter (DAC) joins the output samples. The final inset overlays the analog result (gray): after the start-up transient (t > 0.5 s) the two differ by at most 0.045, and by 0.022 if the sampling rate is doubled. <i>(Python-generated.)</i>

<a name="2.5"></a>
## 2.5 Signals versus Underlying Processes

In most cases we aim to model geophysical phenomena using **deterministic equations** (e.g., acoustic wave equation, Maxwell's equations).  Given a known input, these equations will generate a predictable output (for a noise-less system).  However, distinguishing between the acquired **signal** and the **underlying stochastic process** is often very important.

For example, imagine creating a 1000 point signal by flipping a coin 1000 times. If the coin flip is heads, the corresponding sample is made a value of one. On tails, the sample is set to zero. The process that created this signal has a mean of exactly 0.5, determined by the relative probability of each possible outcome: 50% heads, 50% tails. However, it is unlikely that the actual 1000 point signal will have a mean of exactly 0.5. Random chance will make the number of ones and zeros slightly different each time the signal is generated. The probabilities of the underlying process are constant, but the statistics of the acquired signal change each time the experiment is repeated (Figure 2.4).  This random irregularity found in actual data is called by such names as:  **statistical variation**, **statistical fluctuation**, and **statistical noise**. 

Finally, we live in a world where even though we understand many deterministic equations and systems of equations, the data we record and use are contaminated with both **coherent and incoherent noise**.  Much of the signal processing work that you will do is trying to limit these sources of noise in order to enhance the signal.  This is one of the main reasons why it is good to study digital signal processing!

In [ ]:
C = DSP_COLORS
rng = np.random.default_rng(404)                  # fixed seed: same "random" flips every run
p, N, R = 0.5, 1000, 2000
flips = rng.integers(0, 2, size=N)                # one 1000-sample signal: heads = 1, tails = 0
means = rng.integers(0, 2, size=(R, N)).mean(axis=1)   # sample means of 2000 repeated experiments
sig = np.sqrt(p*(1 - p)/N)
n = np.arange(1, N + 1)
fig, (axa, axb, axc) = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw=dict(width_ratios=[1.2, 1.2, 1]))
ml, sl, bl = axa.stem(n[:60], flips[:60], basefmt='k-')
plt.setp(sl, color=C['sampled'], lw=1); plt.setp(ml, color=C['sampled'], ms=4)
axa.set_xlim(0, 61); axa.set_ylim(-0.1, 1.25); axa.set_xlabel('sample $n$'); axa.set_ylabel('$x[n]$')
axa.set_title('(a) First 60 samples of one signal', fontsize=12)
axb.plot(n, np.cumsum(flips)/n, color=C['time_domain'], lw=1.5, label='running mean of the signal')
axb.fill_between(n, 0.5 - np.sqrt(p*(1 - p)/n), 0.5 + np.sqrt(p*(1 - p)/n), color=C['neutral'], alpha=0.25, lw=0,
                 label=r'$0.5 \pm \sqrt{p(1-p)/n}$')
axb.axhline(0.5, color='k', lw=1, ls='--')
axb.set_xscale('log'); axb.set_xlim(1, N); axb.set_ylim(0, 1); axb.set_xlabel('number of samples $n$'); axb.set_ylabel('mean')
axb.set_title(f'(b) Running mean (final value {flips.mean():.3f})', fontsize=12); axb.legend(loc='upper right', fontsize=9)
axc.hist(means, bins=np.arange(0.4405, 0.5605, 0.005), color=C['noise'], alpha=0.85)
for s_ in (-1, 1):
    axc.axvline(0.5 + s_*sig, color='k', ls='--', lw=1)
axc.set_xlim(0.44, 0.56); axc.set_xlabel('sample mean of 1000 flips'); axc.set_ylabel('number of experiments')
axc.set_title(f'(c) {R} repeated experiments', fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 2.4.</b> A signal versus the process that made it: 1000 fair coin flips (heads = 1). (a) The first 60 samples. (b) The running mean of this one signal wanders and ends at 0.491, not exactly 0.5; the gray band is the expected size of the wander, $\sqrt{p(1-p)/n}$. (c) Sample means of 2000 repeated 1000-flip experiments; the dashed lines are $0.5 \pm \sigma$ with $\sigma = \sqrt{p(1-p)/N} = 0.0158$. A mean at least 0.03 from 0.5 occurs in 5.9% of the experiments (6.2% exactly). <i>(Python-generated.)</i>

<a name="2.6"></a>
## 2.6 DSP versus Analog Signal Processing

It is worth mentioning that one can also perform various [Analog Signal Processing](https://en.wikipedia.org/wiki/Analog_signal_processing) (ASP) tasks.  ASP is any type of signal processing conducted on continuous analog signals by some analog means.  Examples include "bass", "treble" and "volume" controls on stereos, and "tint" controls on TVs.

There are a number of key advantages of DSP over ASP:
* Allows development with use of computers (e.g., with Python, Matlab)
* *Robust tool kits and modularity* - Leverage significant number of complex tools in Python/Matlab without having to redesign physical hardware every time.
* Allows *flexibility* in reconfiguring the DSP operators by changing the program (not the hardware!)
* *Reliable*: processing of 0's and 1's is almost immune to noise, and data are easily stored without deterioration
* *Security* through encryption/scrambling
* *Simplicity* - most operators are additions and subtractions (can be scalar-scalar, vector-scalar, vector-vector, matrix-vector)

However, there are also a number of advantages of ASP over DSP:
* Real-time, high-bandwidth processing with no sampling or computation delay
* No need for data storage and interface with computer CPU

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 2.1 &mdash; How big is a 5D seismic survey?</b>

A 3D seismic survey records traces 4 s long, sampled every 2 ms, on a $100 \times 100$ grid of receivers for each of $50 \times 50$ source positions. Samples are stored as 4-byte floats.

<b>(a)</b> How many time samples are in one trace? What is the size of one shot (one source position) and of the whole data set $D(t, r_x, r_y, s_x, s_y)$?

<b>(b)</b> Each independent variable is one argument of a function of several variables (MATH 213). Which of the five variables are sampled in time and which in space, and which pair would you hold fixed to obtain the shot gather $S(t, r_x, r_y)$ of Section 2.1?

<b>(c)</b> A colleague says a daily closing stock price is a continuous-time signal "because it can take any value". Classify it using Section 2.2, and say what is continuous, what is discrete and what is quantized.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 2.1</b></summary>

<b>(a)</b> $n_t = 4/0.002 + 1 = 2001$ samples. One shot: $2001 \times 100 \times 100 \times 4$ B $= 80.04$ MB. Whole survey: $80.04$ MB $\times 2500$ shots $= 200$ GB ($2.0 \times 10^{11}$ B).

<b>(b)</b> $t$ is sampled in time; $r_x, r_y, s_x, s_y$ are sampled in space. Holding $(s_x, s_y)$ fixed gives one shot gather $S(t, r_x, r_y)$, a 3D signal, like the 2D CAPshot gather of Figure 2.1(b) but with a grid of receivers.

<b>(c)</b> It is a digital signal: time is discrete (one value per trading day) and amplitude is quantized (to the cent). "Can take any value" is not even true of the amplitude, and amplitude is not what decides continuous versus discrete time anyway.
</details>

<div class="alert alert-warning">
<b>Microquestion 2.2 &mdash; When is "discrete" the same as "digital"?</b>

A geophone signal is digitized by an analog-to-digital converter with a full-scale range of $\pm 2.5$ V. Section 2.2 treats discrete-time signals as digital ("the quantizer has infinite resolution"). Consider 16-bit and 24-bit converters.

<b>(a)</b> Compute the quantization step $\Delta = (5\ \mathrm{V})/2^N$ for $N = 16$ and $N = 24$, and the maximum rounding error $\Delta/2$.

<b>(b)</b> A weak microseismic arrival like those in Module 1 produces a peak voltage of 5 μV at the converter (PHGN 200: the geophone coil turns ground velocity into voltage). How many quantization steps does the peak span with each converter? Which converter can record it?

<b>(c)</b> When does the "infinite resolution" assumption of Section 2.2 fail? Use (b) to explain, with reference to Figure 2.2(c).
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 2.2</b></summary>

<b>(a)</b> 16-bit: $\Delta = 5/65\,536 = 7.63 \times 10^{-5}$ V (76.3 μV), maximum error 38.1 μV. 24-bit: $\Delta = 5/16\,777\,216 = 2.98 \times 10^{-7}$ V (0.298 μV), maximum error 0.149 μV.

<b>(b)</b> 16-bit: 5 μV / 76.3 μV = 0.066 steps, so the event rounds to zero and is lost. 24-bit: 5 / 0.298 = 16.8 steps, so it is recorded, though coarsely.

<b>(c)</b> The assumption holds when the signal spans many quantization steps, so the rounding error ($\le \Delta/2$) is small compared with the signal. It fails for signals only a few $\Delta$ in size, as in the 16-bit case of (b), where the quantization error is as large as the signal. In Figure 2.2(c) the signal spans only five levels, so the rounding is easy to see.
</details>

<div class="alert alert-warning">
<b>Microquestion 2.3 &mdash; Signal or process?</b>

Following Section 2.5, a 1000-sample signal is made by flipping a fair coin (heads = 1, tails = 0). The process has mean $p = 0.5$ and variance $p(1-p) = 0.25$ per sample, and the standard deviation of the mean of $N$ independent samples is $\sqrt{p(1-p)/N}$.

<b>(a)</b> Compute the standard deviation of the sample mean for $N = 1000$. How many flips would make it 0.005?

<b>(b)</b> Stacking $N$ repeated recordings averages $N$ copies of the same signal with independent noise. Use the result in (a) to explain why stacking the 40 earthquakes of the CASC93 image in Module 1 improves the noise level by about $\sqrt{40} \approx 6.3$, not 40 (PHGN 100 lab: standard error of the mean).

<b>(c)</b> A student\'s 1000-flip signal has mean 0.53, and they conclude the coin is biased. Critique this using (a) and Figure 2.4(c).
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 2.3</b></summary>

<b>(a)</b> $\sigma = \sqrt{0.25/1000} = 0.0158$. For $\sigma = 0.005$: $N = 0.25/0.005^2 = 10\,000$ flips.

<b>(b)</b> The signal adds coherently (proportional to $N$), while independent noise adds in variance (standard deviation proportional to $\sqrt{N}$), so the signal-to-noise ratio of the average grows as $\sqrt{N}$: $\sqrt{40} = 6.32$.

<b>(c)</b> $0.53 - 0.5 = 0.03 \approx 1.9\sigma$. In Figure 2.4(c) a deviation at least this large occurs in about 6% of repeated fair-coin experiments, so one such result is weak evidence of bias. The statistics of one realization of the signal are not the statistics of the process (Section 2.5).
</details>